In [ ]:
from _init import *

import os
os.environ['CUDA_VISIBLE_DEVICES'] = '1'

In [ ]:
from typing import List
import numpy as np

from ranger.utils import common_utils, json_utils, container_utils
from ranger.corag.corag_result import QueryResult
from ranger.chain_generate.chain_generate_client import request_chain_generate
from ranger.reward.reward_calculator import RewardCalculator

In [ ]:
seed = 42
common_utils.set_seed(seed)

In [ ]:
work_dir = f'/raid/ai/home/jsyang/dev_env/git/repos/ranger'
data_dir = f'{work_dir}/data'
out_dir = f'{work_dir}/output'

test_data_path = f'{data_dir}/custom_multihopqa_eval_1000.jsonl'
test_datas = json_utils.load_file(test_data_path)

In [ ]:
# print(f'{json_utils.to_str(test_datas[0])}')

In [ ]:
def evaluate(prefix, datas, batch_size, n_chains, chain_depth, adapter_path='',
             temperature=-9, top_p=-9, top_k=-9,
             reward_calculator: RewardCalculator=None,
             do_print=True):

    prefix = f'# evaluation_utils.evaluate() {prefix}'.strip()
    data_size = len(datas)

    if DEBUG.EVAL and do_print:
        print(f'{prefix} data_size : {data_size}, batch_size : {batch_size}, n_chains : {n_chains}, chain_depth : {chain_depth}')
        print(f'{prefix} eval start : {common_utils.get_datetime_now()}')
        eval_start = common_utils.get_time_ms()
    
    ems, f1s = [], []
    rewards, advantages = [], []

    for batch_idx, datas_batch in enumerate(container_utils.chunks(datas, batch_size)):
        if DEBUG.EVAL and do_print:
            print(f'{prefix} {batch_idx+1} batch start\t: {common_utils.get_datetime_now()}')
            batch_start = common_utils.get_time_ms()
        
        query_results: List[QueryResult] = request_chain_generate(
            datas_batch,
            batch_size,
            n_chains,
            chain_depth,
            adapter_path,
            temperature,
            top_p,
            top_k,
            is_eval=True
        )

        if reward_calculator is None:
            for query_result in query_results:
                query_result.compute_metrics()
        else:
            reward_calculator.calculate_reward_and_advantage(query_results)

        for query_result in query_results:
            '''
                query_result.compute_metrics() 는 모든 체인을 정답과 비교하여 가장 높은 점수를 query 의 점수로 사용

                greedy(n_chains==1)는 어차피 chain 의 수가 '1'이기 때문에, 상관 없음
                
                단, best-of-n 정답을 보고 가장 높은 체인을 선택하면 안됨
                정답 없이 최선의 체인을 선택하고, 그 체인의 점수를 query 의 점수로 사용해야 함
                CoRAG에서는 모델의 생성 확률을 보고 체인을 선택함

                greedy, best-of-n 둘다 생성 확률이 가장 높은 체인의 스코어를 사용하면 됨
            '''
            log_probs = [chain_result._log_probs[-1] for chain_result in query_result._chain_results]
            max_idx = np.argmax(log_probs)
            
            print(f'log_probs : {log_probs}')
            print(f'max_idx : {max_idx}\n')

            ems.append(query_result._chain_results[max_idx]._em)
            f1s.append(query_result._chain_results[max_idx]._f1)
            rewards.append(query_result._chain_results[max_idx]._reward)
            advantages.append(query_result._chain_results[max_idx]._advantage)
        
        print(f'ems : {ems}')
        print(f'f1s : {f1s}')
        print(f'rewards : {rewards}')
        print(f'advantages : {advantages}')
        return query_results

In [ ]:
batch_size, n_chains, chain_depth = 10, 5, 5
adapter_path = ''
# temperature, top_p, top_k = 0.0, 1.0, -1
temperature, top_p, top_k = 0.7, 0.95, -1

query_results = evaluate(
    '[base]',
    test_datas[:10],
    batch_size,
    n_chains,
    chain_depth,
    adapter_path,
    temperature,
    top_p,
    top_k
)

In [ ]:
for i, query_result in enumerate(query_results):
    for j, chain_result in enumerate(query_result._chain_results):
        print(f'[{i}][{j}] em : {chain_result._em}')
        print(f'[{i}][{j}] f1 : {chain_result._f1}')
        print(f'[{i}][{j}] reward : {chain_result._reward}')
        print(f'[{i}][{j}] advantage : {chain_result._advantage}\n')

In [ ]:
# for query_result in query_results:
#     for chain_result in query_result._chain_results:
#         print(f'{json_utils.to_str(chain_result.to_dict())}')
#         break
#     break